# Module 2 POC Training -- FIRE + LongDRScreening + Tianjin

Runs `train_module2_poc.py` against all three Module 2 data sources at once: FIRE and
LongDRScreening (placeholder Stage-2 conditioning unless a Module 1 cache says otherwise,
same as notebook 04) plus Tianjin (real clinical DR grade conditioning, resolved per-eye --
see notebook 05).

The Generator (`DRForestGAN-v2/base_model.py`) uses AdaIN stage conditioning in its
bottleneck, additive to the original channel-concat conditioning. Training pairs can use a
pre-registered, baseline-aligned follow-up image from `module1/train_registration.py` instead
of the raw one, and after training this notebook runs `evaluate_trajectory.py` to report
per-cascade-step FID/PSNR/SSIM against Tianjin's real follow-up images -- not just a
single-step number. A checkpoint from before the AdaIN change will not load into the current
Generator.

**Round 4** (see the first code cell and the training cell): retina-cropped square images at 256 px,
LongDR's colour photos, registration chosen on FIRE's ground truth (`REGISTRATION_METHOD = 'auto'`),
a frozen grader in the Generator's loss (the EyePACS grader), DiffAugment + a drift penalty on the
critic, best-epoch selection on **stage control** graded by the **other** grader (FGADR), and the
cascade evaluation on **Tianjin patients the Generator never trained on** (`--exclude-train-from`),
graded by the independent grader, also restricted to the manuscript's scope (baseline >= mild NPDR).

**Round 6** (branch `R6-ShapeFix-Controls`): reuses round 5's caches, alignment cache and pretrained
Generator as they are (`INPUT_ROUND_TAG = '_r5'`) and writes only to new `_r6` folders. Retina masks
(`FOV_MASK`: output blacked out outside the baseline's circle; loss and both critic inputs restricted to
baseline ∩ follow-up retina), a gentler fine-tune (lr 1e-5, 40 epochs / patience 10, pretrained critic),
best epoch by KID with the pretrained Generator scored as epoch 0, and a five-row comparison at the end
(copy-the-baseline, round 3, round 5 pretrained, round 5 fine-tuned, round 6) with a keep-or-drop-fine-tuning
verdict. Run order this round: **06, then 08 and 10** (04_05, 09 and 07 are not re-run).

**Round 7** (branch `R7-GradeControl-Eval`): still reads round 5's inputs (`INPUT_ROUND_TAG = '_r5'`), writes only to
new `_r7` folders. Round 6's Generator ignored the requested grade (stage control 0 % every epoch), so round 7 (1) puts
the frozen EyePACS grader back in G's loss at half round 4's weight (`LAMBDA_GRADER = 0.5`) and lowers the
reconstruction weight (`LAMBDA_REC = 5`); (2) picks the best epoch only among epochs with stage control >= 20 % AND KID
within 2x the pretrained Generator's (`SELECTION_MODE = 'control-kid'`; none -> the pretrained Generator); (3) adds a
**real follow-up** ceiling row, a **"grade went up"** column and a **second grader** (FGADR) to the comparison; (4)
shrinks/softens the retina mask and colour-matches every cascade step to the baseline (`FOV_ERODE`, `FOV_FEATHER`,
`COLOR_MATCH`), plus a row that generates every stage straight from the baseline. Run order: **06, then 08 and 10**.

**Run this after notebook 04** (FIRE/LongDR Module 1 caches) **and notebook 05** (Tianjin
Module 1 cache + laterality resolution) **have both finished** -- this notebook only reads
their cache outputs from Drive, it doesn't rebuild them.

**EyePACS pretraining:** `USE_NONLONG_PRETRAIN = True` (default, cell before training) fine-tunes
from notebook 09's non-longitudinal pretrained weights (run `models_poc_tianjin_from_nonlong`);
set it False only for the random-init ablation (`models_poc_tianjin`). **`MODULE1_SOURCE`**
(first code cell) must match notebook 04_05's; every run name gets that setting's suffix
(`_drgnet_comb` for the default `'drgnet_combined'`, `_drgnet` for `'drgnet'`). Checkpoints are copied to `Thesis_Datasets/module2_runs/` on Drive for notebook 10.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
# Free Drive space: the training notebooks save checkpoints and caches here (several GB in total).
# Deleted or overwritten files stay in the Drive Trash and still count -- empty it if space is low.
import shutil as _shutil
try:
    _du = _shutil.disk_usage('/content/drive/MyDrive')
    print(f'Google Drive: {_du.free / 1e9:.1f} GB free of {_du.total / 1e9:.1f} GB')
    if _du.free < 10e9:
        print('⚠ Under 10 GB free on Google Drive -- model files may fail to save. Empty the Trash '
              '(drive.google.com/drive/trash) or free space before training.')
except OSError:
    pass

import getpass, os
GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

REPO_OWNER = 'Mieka068'
REPO_NAME = 'DRProgression'
REPO_BRANCH = 'R7-GradeControl-Eval'  # <-- round 7; 'R6-ShapeFix-Controls' = round 6, 'R5-Round3Baseline-Fixes' = round 5, 'AllModuleEdits-PreFinals' = rounds 3/4, 'mod-2-NonLongIntegration' = the earlier runs
REPO_CODE_SUBDIR = 'M2-DRProgression-VerM-module1-fgadr-poc'

DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'
assert os.path.isdir(DRIVE_DATA_DIR), f"Not found: {DRIVE_DATA_DIR}"

# Which Module 1 models the pipeline uses -- set the SAME value in notebooks 04_05 and 06-10:
#   'drgnet' -- notebook 11's DRG-Net replication checkpoints (module1_runs/drgnet_replication/)
#   'poc'    -- notebooks 02/03's original POC checkpoints (module1_runs/)
# Everything built from Module 1's outputs (Module 1 caches, Module 2 runs, figures) gets a
# '_drgnet' suffix for 'drgnet', so the two pipelines never overwrite each other's results.
MODULE1_SOURCE = 'drgnet_combined'
assert MODULE1_SOURCE in ('drgnet_combined', 'drgnet', 'poc'), MODULE1_SOURCE
# New Drive folders for this branch's runs: every Module 1 cache / Module 2 / Module 3 folder below
# gets RUN_SUFFIX, so the mod-2-NonLongIntegration results stay untouched for before/after
# comparison. '' = read/write the earlier folders again.
RUN_SUFFIX = '_prefinals'
RUN_TAG = {'drgnet_combined': '_drgnet_comb', 'drgnet': '_drgnet', 'poc': ''}[MODULE1_SOURCE] + RUN_SUFFIX
# Round 4: every Module 1 cache / Module 2 / Module 3 folder ALSO gets ROUND_TAG, so round 3's results
# (no ROUND_TAG) stay untouched for before/after comparison -- the SAME value in notebooks 04_05 and 06-10.
# Model folders (notebook 11) and the laterality file keep RUN_SUFFIX only. '' = round 3's folders.
# Round 5 (branch R5-Round3Baseline-Fixes): round 3's Module 2 settings (128 px, no grader loss, round 3's
# best-epoch rule) + the data fixes (text-safe retina crop, SIFT failures dropped). '_r4' = round 4's folders.
# Round 6 (branch R6-ShapeFix-Controls): only Module 2's fine-tuning and evaluation change. Everything round 6
# READS comes from round 5's folders as they are (INPUT_ROUND_TAG: Module 1 caches, laterality file, SIFT
# alignment cache, notebook 09's pretrained Generator); everything it WRITES goes to new '_r6' folders
# (ROUND_TAG). Notebooks 04_05, 09 and 07 are not re-run. Rounds 3-5 stay untouched.
# Round 7 (branch R7-GradeControl-Eval): same inputs as round 6 (round 5's), outputs to new '_r7' folders.
ROUND_TAG = '_r7'          # outputs: this round's Module 2 run, evaluations, figures
INPUT_ROUND_TAG = '_r5'    # inputs reused unchanged from round 5
BASE_RUN_TAG = RUN_TAG
INPUT_RUN_TAG = BASE_RUN_TAG + INPUT_ROUND_TAG
RUN_TAG = BASE_RUN_TAG + ROUND_TAG
# Image preprocessing for every script this notebook runs (fundus_preprocess.py), the same in 04_05-10:
#   DRP_RETINA_CROP '1' = crop to the retina and pad to square before resizing (Tianjin's 3:2 and LongDR's
#     2000x1312 photos were squashed into ovals); '0' = round 3's plain resize.
#   DRP_LONGDR_FOLDER 'FundusImagesColor' = LongDR's raw colour photos. Round 3 used FundusImagesNormalized,
#     on which the lesion segmenters (trained on raw photos) marked 10-30 % of a NORMAL retina as lesion
#     (LBS 0.1-0.34 at grade 0, vs ~0.001 on Tianjin/FIRE) -- 559 of the 917 Module 2 pairs.
os.environ['DRP_RETINA_CROP'] = '1'
os.environ['DRP_LONGDR_FOLDER'] = 'FundusImagesColor'
# Module 2 resolution -- the SAME value in notebooks 06, 08, 09 and 10 (item 4.3; the scripts stop
# with an error on a mismatch).
# Round 4: 'auto' = the size notebook 04_05's grader-resolution check recommends (its
# grader_resolution_check.json in this round's Module 1 cache folder): 256 px when the grader reads real
# Tianjin images clearly better at 256 than at 128 px (kappa +0.05 or more), else 128 (~4x faster). Set a
# number (128 / 256) to override. Run 04_05 to the end BEFORE 09 / 06, so every notebook gets the same size.
MODULE2_IMAGE_SIZE = 128  # round 5: 128 px (round 3) until the outputs are clean; 'auto' = notebook 04_05's recommendation (256 in round 4)
MODULE1_RUNS = os.path.join(DRIVE_DATA_DIR, 'module1_runs', *(['drgnet_replication'] if MODULE1_SOURCE != 'poc' else []))
# Notebook 11 on this branch writes its models to drgnet_replication<RUN_SUFFIX>/ (tiled evaluation,
# 200 segmentation epochs, optional EyePACS grader); use those once they exist.
if MODULE1_SOURCE != 'poc' and os.path.isdir(os.path.join(MODULE1_RUNS + RUN_SUFFIX, 'segmentation')):
    MODULE1_RUNS += RUN_SUFFIX
elif MODULE1_SOURCE != 'poc' and RUN_SUFFIX:
    print(f'\u26a0 {MODULE1_RUNS + RUN_SUFFIX} not found -- using the earlier Module 1 models in {MODULE1_RUNS}. '
          'Run notebook 11 on this branch first for the re-trained ones.')
# Lesion segmenters: <MODULE1_RUNS>/segmentation/<SEG_MODEL_PREFIX><lesion>/model_2.pth.tar --
# FGADR + DDR combined (notebook 11) or FGADR only. The classifier is FGADR's either way.
SEG_MODEL_PREFIX = 'comb_models_' if MODULE1_SOURCE == 'drgnet_combined' else 'models_FGADR_NO_TATL_'
# Round 4: notebook 11's fine-tuned combined segmenters (comb_ft_models_*) once all four exist.
if MODULE1_SOURCE == 'drgnet_combined':
    if all(os.path.isfile(os.path.join(MODULE1_RUNS, 'segmentation', f'comb_ft_models_{_l}', 'model_2.pth.tar'))
           for _l in ('ex', 'he', 'ma', 'se')):
        SEG_MODEL_PREFIX = 'comb_ft_models_'
    elif ROUND_TAG:
        print('\u26a0 Fine-tuned segmenters (comb_ft_models_*) not all on Drive yet -- using the 200-epoch '
              'comb_models_*. Run notebook 11 (RUN["comb_seg_ft"]) first for round 4.')
print('Lesion segmenters:', SEG_MODEL_PREFIX + '<lesion>')
MODULE1_CACHE_DIR = os.path.join(DRIVE_DATA_DIR, 'module1_cache' + INPUT_RUN_TAG)  # round 5's caches (read only)
os.makedirs(MODULE1_CACHE_DIR, exist_ok=True)
print('Module 1 source:', MODULE1_SOURCE, '| models:', MODULE1_RUNS, '| caches:', MODULE1_CACHE_DIR)
if MODULE2_IMAGE_SIZE == 'auto':
    import json as _json
    _res_json = os.path.join(MODULE1_CACHE_DIR, 'grader_resolution_check.json')
    if os.path.isfile(_res_json):
        _res = _json.load(open(_res_json))
        if _res.get('recommended_image_size'):
            MODULE2_IMAGE_SIZE = int(_res['recommended_image_size'])
        else:  # a check file written before the recommended_image_size field
            _gain = max((_res.get('kappa_gain_high_vs_low') or {0: 0}).values())
            MODULE2_IMAGE_SIZE = max(_res['sizes']) if _gain >= 0.05 else min(_res['sizes'])
        print(f'MODULE2_IMAGE_SIZE = {MODULE2_IMAGE_SIZE} (auto, from {_res_json}: {_res.get("recommendation")})')
    else:
        MODULE2_IMAGE_SIZE = 256
        print(f'\u26a0 MODULE2_IMAGE_SIZE = 256 (auto fallback): {_res_json} not found -- run notebook 04_05 to the end '
              'first, or set MODULE2_IMAGE_SIZE by hand. Every Module 2 notebook must use the same size.')
CACHE_DIR = MODULE1_CACHE_DIR
for _name in ['module1_outputs_fire.pt', 'module1_outputs_longdr.pt', 'module1_outputs_tianjin.pt']:
    _p = os.path.join(CACHE_DIR, _name)
    assert os.path.isfile(_p), (
        f"Not found: {_p} -- run notebook 04_05 first (with the same MODULE1_SOURCE)."
    )
    print('✓', _p)

Mounted at /content/drive
Google Drive: 151.8 GB free of 214.7 GB
GitHub Personal Access Token (repo scope): ··········
Lesion segmenters: comb_ft_models_<lesion>
Module 1 source: drgnet_combined | models: /content/drive/MyDrive/Thesis_Datasets/module1_runs/drgnet_replication_prefinals | caches: /content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5
✓ /content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5/module1_outputs_fire.pt
✓ /content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5/module1_outputs_longdr.pt
✓ /content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5/module1_outputs_tianjin.pt


In [2]:
# Unzip the three raw datasets locally (train_module2_poc.py's loaders read images directly
# at training time, not just at cache-build time). Same unzip conventions as notebooks 01/04/05.
import glob, shutil

os.makedirs('/content/data', exist_ok=True)
%cd /content/data
!unzip -q -n "$DRIVE_DATA_DIR/FIRE_dataset.zip"
!unzip -q -n "$DRIVE_DATA_DIR/LongDRScreening_20150209.zip" -d LongDRScreening_20150209
!unzip -q -n "$DRIVE_DATA_DIR/retinal-dr-longitudinal.zip" -d _tianjin_extract_raw

_manifest_candidates = glob.glob('/content/data/_tianjin_extract_raw/**/corrected_manifest.csv', recursive=True)
assert _manifest_candidates, 'No corrected_manifest.csv found -- see notebook 05 for troubleshooting.'
_tianjin_root = os.path.dirname(_manifest_candidates[0])
TIANJIN_DIR = '/content/data/retinal-dr-longitudinal'
if _tianjin_root != TIANJIN_DIR and not os.path.exists(TIANJIN_DIR):
    os.symlink(_tianjin_root, TIANJIN_DIR)

print('FIRE Images present   :', os.path.isdir('/content/data/FIRE_dataset/FIRE/Images'))
print('LongDR norm present   :', os.path.isdir('/content/data/LongDRScreening_20150209/FundusImagesNormalized'))
print(f"LongDR images used    : {os.environ['DRP_LONGDR_FOLDER']} (present: "
      f"{os.path.isdir(os.path.join('/content/data/LongDRScreening_20150209', os.environ['DRP_LONGDR_FOLDER']))})")
print('Tianjin manifest found:', os.path.isfile(os.path.join(TIANJIN_DIR, 'corrected_manifest.csv')))

/content/data
FIRE Images present   : True
LongDR norm present   : True
LongDR images used    : FundusImagesColor (present: True)
Tianjin manifest found: True


In [3]:
# Clone this repo. Skips cleanly if already cloned in this runtime.
%cd /content
if not os.path.isdir(f'/content/{REPO_NAME}'):
    !git clone --branch {REPO_BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_NAME}

REPO_CODE_DIR = f'/content/{REPO_NAME}/{REPO_CODE_SUBDIR}'
assert os.path.isdir(REPO_CODE_DIR), f"Not found: {REPO_CODE_DIR} -- check REPO_BRANCH/REPO_CODE_SUBDIR above"

%cd {REPO_CODE_DIR}
!pip install -q pandas openpyxl segmentation-models-pytorch torchmetrics torch-fidelity opencv-python-headless

/content
Cloning into 'DRProgression'...
remote: Enumerating objects: 1248, done.
remote: Counting objects: 100% (414/414), done.
remote: Compressing objects: 100% (255/255), done.
remote: Total 1248 (delta 260), reused 228 (delta 159), pack-reused 834 (from 1)
Receiving objects: 100% (1248/1248), 44.77 MiB | 17.25 MiB/s, done.
Resolving deltas: 100% (745/745), done.
/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc


In [4]:
# Laterality resolution: Organized_Data of Patients.xlsx grades DR per eye (OS/OD), not per
# patient, and corrected_manifest.csv doesn't say which eye a row is -- tianjin_dataset.py /
# module3/dataset.py both require laterality_resolved.csv to exist in TIANJIN_DIR before they
# can run. Reuse a Drive-persisted copy if one already exists (resolving ~1,100 images takes a
# few minutes); otherwise compute it fresh here and persist it to Drive for next time.
# This branch re-tuned module1/resolve_eye_laterality.py (item 7.1), so it gets its own copy
# (module1_cache<RUN_SUFFIX>/); the settings come from notebook 11's EyePACS validation when it exists.
# _v2: the first file on this branch used the EyePACS validation's convention at only ~61 % accuracy
# (EyePACS mixes mirrored and non-mirrored images) -> 20.6 % of patients had both eyes labelled alike.
# Round 5: its own copy in this round's cache folder (the text-safe crop changes the field of view the
# optic-disc position is measured in); rounds 3/4 ('' / '_r4') share module1_cache<RUN_SUFFIX>/.
_LAT_DIR = 'module1_cache' + (RUN_SUFFIX if INPUT_ROUND_TAG in ('', '_r4') else INPUT_RUN_TAG)  # round 6: round 5's file
LATERALITY_DRIVE_PATH = os.path.join(DRIVE_DATA_DIR, _LAT_DIR, 'laterality_resolved_v2.csv')
os.makedirs(os.path.dirname(LATERALITY_DRIVE_PATH), exist_ok=True)
LATERALITY_VALIDATION = os.path.join(DRIVE_DATA_DIR, 'module1_cache' + RUN_SUFFIX, 'laterality_eyepacs_validation.json')
LATERALITY_LOCAL_PATH = os.path.join(TIANJIN_DIR, 'laterality_resolved.csv')

if os.path.isfile(LATERALITY_DRIVE_PATH):
    import shutil
    shutil.copy(LATERALITY_DRIVE_PATH, LATERALITY_LOCAL_PATH)
    print(f'✓ Reused laterality_resolved.csv from Drive: {LATERALITY_DRIVE_PATH}')
else:
    %cd {REPO_CODE_DIR}
    # LATERALITY_DISC_LEFT_IS: which eye a left-of-centre optic disc means in Tianjin's images. None =
    # the script's default ('OD', what every earlier run used), unless notebook 11's EyePACS validation
    # is clearly reliable (>= 80 % accuracy). Check a few Tianjin images of known right eyes by eye and
    # set 'OD' or 'OS' here -- that is the real answer (after changing it, delete laterality_resolved_v2.csv
    # on Drive so it is recomputed).
    LATERALITY_DISC_LEFT_IS = 'OS'  # round 3 set this by hand after checking Tianjin images by eye
    _lat_args = ''
    _b = None
    if os.path.isfile(LATERALITY_VALIDATION):
        import json as _json
        _b = _json.load(open(LATERALITY_VALIDATION))['best']
    if LATERALITY_DISC_LEFT_IS:
        _lat_args = f' --disc-left-is {LATERALITY_DISC_LEFT_IS}'
        print(f'Laterality: --disc-left-is {LATERALITY_DISC_LEFT_IS} (set by hand)')
    elif _b and _b['accuracy_on_decided'] >= 0.8:
        _lat_args = (f" --disc-left-is {_b['disc_left_is']} --margin {_b['margin']}"
                     + (' --legacy-estimator' if _b['estimator'] == 'legacy' else ''))
        print(f"Laterality settings from the EyePACS validation: {_lat_args} "
              f"(accuracy {_b['accuracy_on_decided']:.1%}, {_b['uncertain_rate']:.1%} uncertain)")
    else:
        print('\u26a0 Laterality: script defaults (--disc-left-is OD --margin 0.1) -- '
              + (f"the EyePACS validation is too weak to use ({_b['accuracy_on_decided']:.1%} accuracy). "
                 if _b else 'no EyePACS validation. ')
              + 'Set LATERALITY_DISC_LEFT_IS after checking a few Tianjin images by eye.')
    !python module1/resolve_eye_laterality.py --dataset-dir "{TIANJIN_DIR}"{_lat_args}
    # Read the printed same-eye disagreement rate above before trusting this for training --
    # see resolve_eye_laterality.py's module docstring (>5% disagreement means the heuristic
    # needs tuning).
    os.makedirs(os.path.dirname(LATERALITY_DRIVE_PATH), exist_ok=True)
    import shutil
    shutil.copy(LATERALITY_LOCAL_PATH, LATERALITY_DRIVE_PATH)
    print(f'✓ Computed and persisted laterality_resolved.csv to Drive: {LATERALITY_DRIVE_PATH}')

✓ Reused laterality_resolved.csv from Drive: /content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5/laterality_resolved_v2.csv


In [5]:
# Trains the training-only affine registration network and pre-warps every baseline<->follow-up
# pair (all 3 sources) so train_module2_poc.py can condition on baseline-aligned follow-up images.
# Saved to Drive and REUSED on later sessions: registration training is random, so regenerating it
# would hand a resumed Module 2 run different aligned images than it started with. Delete the
# folder below to rebuild it (e.g. after the datasets change). Independent of MODULE1_SOURCE.
# On this branch (items 5.1/5.2): Tianjin pairs whose recovered pairing quality is below
# TIANJIN_MIN_PAIR_QUALITY are dropped (registration AND training); every run first validates the
# warps against FIRE's ground-truth control points (identity vs. network vs. SIFT + RANSAC, printed
# and saved as fire_registration_validation.json). REGISTRATION_METHOD = 'sift' builds the cache with
# the classical registration instead -- use it if the network barely beats identity. Each setting
# gets its own folder (RUN_SUFFIX, method, size), so the earlier registration/ cache is kept.
TIANJIN_MIN_PAIR_QUALITY = 0.5   # None = keep every pair (earlier behaviour)
# Round 4: 'auto' (default) trains the network, scores identity / net / SIFT on FIRE's ground truth (SIFT now
# in the same retina-cropped frame, implausible SIFT warps rejected) and builds the cache with whichever
# has the lowest median error -- 'identity' (no warp) when neither beats it (round 3: net 44.8 px vs no
# warp 44.2 px). The choice is in fire_registration_validation.json ('cache_method'); report it.
# Round 5: 'sift' (best on FIRE in round 4: 11.4 px vs 44.2 px unaligned). Pairs SIFT can't align are DROPPED
# from training (round 4 kept 455 of them raw). Read the 'SIFT failure rate' line below: over ~15 %, set
# 'identity' (round 3's behaviour: no warping) for this round instead.
REGISTRATION_METHOD = 'sift'     # 'auto', 'net' (self-supervised affine network), 'sift' or 'identity'
# Round 6: round 5's SIFT cache, reused as it is (INPUT_ROUND_TAG) -- not rebuilt.
REG_DIR = os.path.join(DRIVE_DATA_DIR, 'module2_runs', 'registration' + RUN_SUFFIX + INPUT_ROUND_TAG
                       + ('' if REGISTRATION_METHOD == 'net' else f'_{REGISTRATION_METHOD}')
                       + ('' if MODULE2_IMAGE_SIZE == 128 else f'_{MODULE2_IMAGE_SIZE}'))
QUALITY_ARGS = f' --tianjin-min-pair-quality {TIANJIN_MIN_PAIR_QUALITY}' if TIANJIN_MIN_PAIR_QUALITY is not None else ''
REGISTRATION_CACHE = os.path.join(REG_DIR, 'registered_followups.pt')
if os.path.isfile(REGISTRATION_CACHE):
    print('✓ Reusing registration cache from Drive:', REGISTRATION_CACHE)
elif INPUT_ROUND_TAG != ROUND_TAG:
    raise FileNotFoundError(f'{REGISTRATION_CACHE} not found -- round 6 reuses round 5\'s alignment cache '
                            f'(INPUT_ROUND_TAG = {INPUT_ROUND_TAG!r}); check that folder on Drive')
else:
    os.makedirs(REG_DIR, exist_ok=True)
    %cd {REPO_CODE_DIR}/module1
    !python train_registration.py --image-size {MODULE2_IMAGE_SIZE} \
        --fire-dir /content/data/FIRE_dataset \
        --longdr-dir /content/data/LongDRScreening_20150209 \
        --tianjin-dir "{TIANJIN_DIR}"{QUALITY_ARGS} \
        --num-epochs 10 --method {REGISTRATION_METHOD} \
        --checkpoint-out "{REG_DIR}/registration_net.ckpt" \
        --validation-out "{REG_DIR}/fire_registration_validation.json" \
        --cache-out "{REGISTRATION_CACHE}"
import json as _json
_v = os.path.join(REG_DIR, 'fire_registration_validation.json')
if os.path.isfile(_v):
    _v = _json.load(open(_v))
    print('Registration on FIRE (median error, original px):',
          {m: round(_v[m]['median_error'], 1) for m in ('identity', 'net', 'sift') if m in _v},
          '-> cache built with', _v.get('cache_method'))
_sr = os.path.splitext(REGISTRATION_CACHE)[0] + '_sift_report.json'
if os.path.isfile(_sr):
    _sr = _json.load(open(_sr))
    print(f"SIFT failure rate: {_sr['sift_failed']}/{_sr['pairs']} = {_sr['failure_rate']:.1%} "
          f"(failed pairs {_sr['failed_pairs']}) -- per source: {_sr['per_source']}")
    if _sr['failure_rate'] > 0.15:
        print('⚠ Over 15 % of pairs failed SIFT: set REGISTRATION_METHOD = "identity" and re-run this cell.')

✓ Reusing registration cache from Drive: /content/drive/MyDrive/Thesis_Datasets/module2_runs/registration_prefinals_r5_sift/registered_followups.pt
Registration on FIRE (median error, original px): {'identity': 44.2, 'sift': 11.4} -> cache built with sift
SIFT failure rate: 246/917 = 26.8% (failed pairs dropped) -- per source: {'FIRE': {'pairs': 14, 'sift_failed': 2}, 'LongDRScreening': {'pairs': 559, 'sift_failed': 230}, 'Tianjin': {'pairs': 344, 'sift_failed': 14}}
⚠ Over 15 % of pairs failed SIFT: set REGISTRATION_METHOD = "identity" and re-run this cell.


In [6]:
# Locate the same Module 1 checkpoints notebooks 04/05 already found on Drive -- the cascade
# re-segments/re-grades each synthesized step with Module 1 itself (not just the GAN), so
# evaluate_trajectory.py needs these too, not just the module1_cache .pt files. Before training on
# this branch: the grader also scores every epoch's val outputs for best-epoch selection (item 5.3).
import glob

SEG_DIR   = os.path.join(MODULE1_RUNS, 'segmentation')
# Module 1 grader: 'eyepacs' (default) = DRG-Net's classifier trained on EyePACS (notebook 11, training
# capped at 3000 images/grade); 'fgadr' = the FGADR grader. Notebook 04_05's grader-comparison cell scores
# both on Tianjin's clinical grades -- set the one it recommends in notebooks 04_05, 06, 08, 09 and 10.
# With 'eyepacs' and no EyePACS grader on Drive, the FGADR one is used.
CLASSIFIER_RUN = 'eyepacs'
if not glob.glob(os.path.join(MODULE1_RUNS, 'classification', CLASSIFIER_RUN, 'saves', '*.pt')):
    print(f'\u26a0 No {CLASSIFIER_RUN} grader in {MODULE1_RUNS} yet -- using the FGADR one')
    CLASSIFIER_RUN = 'fgadr'
CLS_SAVES = os.path.join(MODULE1_RUNS, 'classification', CLASSIFIER_RUN, 'saves')
# Classifier input normalisation must match its training config (eyepacs_poc.yaml differs).
CLS_NORM_ARGS = ' --classifier-norm eyepacs' if CLASSIFIER_RUN == 'eyepacs' else ''

# EyePACS: best_validation_weights.pt (picked on the Kaggle public test, kept apart from its test set);
# FGADR: final_weights.pt (its "validation" set IS its test set, so the best-validation pick is optimistic).
_pref  = (['best_validation_weights.pt', 'final_weights.pt'] if CLASSIFIER_RUN == 'eyepacs'
          else ['final_weights.pt', 'best_validation_weights.pt'])
_cands = [os.path.join(CLS_SAVES, n) for n in _pref] + sorted(
    glob.glob(os.path.join(CLS_SAVES, '*.pt')), key=os.path.getmtime, reverse=True)
CLS_CKPT = next((p for p in _cands if os.path.isfile(p)), None)
assert CLS_CKPT, f"No classifier .pt in {CLS_SAVES} -- run notebook 02/03 (MODULE1_SOURCE='poc') or 11 ('drgnet' / 'drgnet_combined') first"

EX_CKPT = os.path.join(SEG_DIR, SEG_MODEL_PREFIX + 'ex', 'model_2.pth.tar')
MA_CKPT = os.path.join(SEG_DIR, SEG_MODEL_PREFIX + 'ma', 'model_2.pth.tar')
for _p in (CLS_CKPT, EX_CKPT, MA_CKPT):
    assert os.path.isfile(_p), f"missing checkpoint: {_p} -- run notebook 02/03 (MODULE1_SOURCE='poc') or 11 ('drgnet' / 'drgnet_combined') first"
print('classifier :', CLS_CKPT)
# Round 4 (item R4-3.1/3.3): CLS_CKPT (CLASSIFIER_RUN, the EyePACS grader by default) is the frozen grader in
# the Generator's loss. Best-epoch selection and the cascade evaluation use the OTHER grader (INDEP_CKPT), so
# the reported agreement is not the number the Generator was optimised on.
INDEP_RUN = 'fgadr' if CLASSIFIER_RUN == 'eyepacs' else 'eyepacs'
_ind = os.path.join(MODULE1_RUNS, 'classification', INDEP_RUN, 'saves')
INDEP_CKPT = next((os.path.join(_ind, n) for n in (['final_weights.pt', 'best_validation_weights.pt'] if INDEP_RUN == 'fgadr'
                                                   else ['best_validation_weights.pt', 'final_weights.pt'])
                   if os.path.isfile(os.path.join(_ind, n))), None)
INDEP_NORM = INDEP_RUN
CLS_NORM = 'eyepacs' if CLASSIFIER_RUN == 'eyepacs' else 'fgadr'
if INDEP_CKPT is None:
    print(f'\u26a0 No {INDEP_RUN} grader in {MODULE1_RUNS} -- selection/evaluation fall back to the grader in the '
          'loss, so the reported agreement is optimistic. Train it in notebook 11.')
    INDEP_CKPT, INDEP_NORM = CLS_CKPT, CLS_NORM
print('independent grader (selection + evaluation):', INDEP_CKPT, f'({INDEP_NORM})')
print('EX seg     :', EX_CKPT)
print('MA seg     :', MA_CKPT)
# HE/SE checkpoints -- included automatically once they exist on Drive, omitted otherwise
# (apply_to_progression_data.py/evaluate_trajectory.py both leave a lesion channel out rather
# than fail if no checkpoint is passed for it).
HE_CKPT = os.path.join(SEG_DIR, SEG_MODEL_PREFIX + 'he', 'model_2.pth.tar')
SE_CKPT = os.path.join(SEG_DIR, SEG_MODEL_PREFIX + 'se', 'model_2.pth.tar')
print('HE seg     :', HE_CKPT if os.path.isfile(HE_CKPT) else 'not trained yet')
print('SE seg     :', SE_CKPT if os.path.isfile(SE_CKPT) else 'not trained yet')

classifier : /content/drive/MyDrive/Thesis_Datasets/module1_runs/drgnet_replication_prefinals/classification/eyepacs/saves/best_validation_weights.pt
independent grader (selection + evaluation): /content/drive/MyDrive/Thesis_Datasets/module1_runs/drgnet_replication_prefinals/classification/fgadr/saves/final_weights.pt (fgadr)
EX seg     : /content/drive/MyDrive/Thesis_Datasets/module1_runs/drgnet_replication_prefinals/segmentation/comb_ft_models_ex/model_2.pth.tar
MA seg     : /content/drive/MyDrive/Thesis_Datasets/module1_runs/drgnet_replication_prefinals/segmentation/comb_ft_models_ma/model_2.pth.tar
HE seg     : /content/drive/MyDrive/Thesis_Datasets/module1_runs/drgnet_replication_prefinals/segmentation/comb_ft_models_he/model_2.pth.tar
SE seg     : /content/drive/MyDrive/Thesis_Datasets/module1_runs/drgnet_replication_prefinals/segmentation/comb_ft_models_se/model_2.pth.tar


In [7]:
# Start Module 2 from notebook 09's EyePACS (non-longitudinal) pretraining -- the method.
# Pretraining ablation (item 5.7): after the main run, set this False and run the notebook again
# (Runtime -> Run all). That run (random init, models_poc_tianjin{RUN_TAG}) has its own Drive folder,
# samples and figures, so the two runs' trajectory_eval_results.json compare directly.
USE_NONLONG_PRETRAIN = True

NONLONG_DIR = os.path.join(DRIVE_DATA_DIR, 'module2_runs', 'nonlong_pretrain' + INPUT_RUN_TAG)  # round 5's pretraining (read only)
if USE_NONLONG_PRETRAIN:
    for _f in ('final-G.ckpt', 'final-D.ckpt'):
        assert os.path.isfile(os.path.join(NONLONG_DIR, _f)), (
            f"{NONLONG_DIR}/{_f} missing -- run notebook 09 first (same MODULE1_SOURCE), "
            "or set USE_NONLONG_PRETRAIN = False")
    INIT_ARGS = (f' --init-generator "{NONLONG_DIR}/final-G.ckpt"'
                 f' --init-discriminator "{NONLONG_DIR}/final-D.ckpt"')
    MODULE2_RUN_NAME = 'models_poc_tianjin_from_nonlong' + RUN_TAG
else:
    INIT_ARGS = ''
    MODULE2_RUN_NAME = 'models_poc_tianjin' + RUN_TAG
print('Module 2 run:', MODULE2_RUN_NAME, '| init:', INIT_ARGS or 'random')


Module 2 run: models_poc_tianjin_from_nonlong_drgnet_comb_prefinals_r7 | init:  --init-generator "/content/drive/MyDrive/Thesis_Datasets/module2_runs/nonlong_pretrain_drgnet_comb_prefinals_r5/final-G.ckpt" --init-discriminator "/content/drive/MyDrive/Thesis_Datasets/module2_runs/nonlong_pretrain_drgnet_comb_prefinals_r5/final-D.ckpt"


In [8]:
# --fire-dir / --longdr-dir / --tianjin-dir MUST be passed explicitly: train_module2_poc.py's
# own argparse defaults resolve relative to the cloned repo's working directory, not where the
# data actually lives on the Colab VM (see combined_dataset.py / train_module2_poc.py docstrings).
# --eval-batches raised from the script's own unexamined default of 5 to 15 (with the
# default batch-size=4, that's ~60 samples, above min-n-for-fid=50) so FID/KID actually
# compute rather than reporting None -- see IMPLEMENTATION_PLAN_3.md Task L and
# image_quality_metrics.py. Raise further if the combined dataset has more to spare.
#
# MODULE2_EPOCHS: 100 (was 50). With n_critic = 5 (train_module2_poc.py's default on this branch) the
# Generator is updated once per 5 batches, so it needs more epochs for the same number of G steps
# (printed as 'G steps'); go to 200 only once the val curve shows training still helping. Training state is saved every epoch on the runtime's disk and copied to
# Drive (resume_state.pt in this run's Drive folder) every 10 epochs and after the last one --
# rewriting it on Drive every epoch would fill the Drive with old copies. If Colab disconnects,
# re-run this notebook: training continues from the last Drive copy (at most 9 epochs redone).
# Raising MODULE2_EPOCHS later and re-running also continues the same run up to the new total.
# To start over, delete that resume_state.pt and restart the runtime.
#
# Best-epoch saving: 10% of the eyes/patients (--val-fraction 0.1) are held out of training;
# after every epoch the Generator is scored on them (L1 between its synthesized follow-up and
# the real one) and the best epoch is saved as best-G.ckpt -- that is the model every later
# step uses (evaluation below, notebooks 08 and 10). final-G.ckpt is the last epoch. The
# printed "val L1 ... new best" lines show which epoch won.
MODULE2_EPOCHS = 40  # round 6: 40 (was 100) -- gentler fine-tune from the pretrained Generator
# Best epoch (item 5.3) = lowest val L1 - SELECTION_ALPHA x Module 1 agreement with the target stage on
# the val pairs' synthesized follow-ups (CLS_CKPT above; SELECTION_ALPHA = 0 -> val L1 only, the earlier
# rule). EARLY_STOP_PATIENCE: stop after that many epochs without a better score (0 = run all epochs).
# Stage 0 baselines (item 7.2): 0 keeps every baseline grade (the default -- more data, documented as
# in scope); 1 keeps only eyes that start at >= mild NPDR (the manuscript's scope). Use the same value
# in notebooks 06 and 07.
MIN_BASELINE_GRADE = 0
SELECTION_ALPHA = 0.1
EARLY_STOP_PATIENCE = 15  # round 7: 15 -- the grader loss needs a few epochs to move stage control (round 6: 10, before: 25)
# Round 4 (train_module2_poc.py docstring point 14, gan_guidance.py):
#   LAMBDA_GRADER  frozen-grader loss weight (CLS_CKPT, augmented views); 0 = off (round 3)
#   DIFFAUGMENT    critic augmentation ('' = off) + DRIFT drift penalty -- the critic over-fitted the 35
#                  progressing pairs in round 3 (best epoch 11, D loss negative from epoch 34)
#   SELECTION_MODE 'controllability' = best epoch by stage control (one stage above each held-out pair's
#                  baseline, graded by INDEP_CKPT); 'target' = round 3's agreement with the target grade
# Round 5: round 3's settings -- no grader loss (it painted green/yellow marks the grader read as lesions),
# no DiffAugment / drift penalty (round 4 additions), best epoch by agreement with the target grade.
LAMBDA_GRADER = 0.0
DIFFAUGMENT = ''
DRIFT = 0.0
SELECTION_MODE = 'target'
# Round 6 (train_module2_poc.py docstring point 15):
#   FOV_MASK      G's output is blacked out outside the baseline's retina circle; the reconstruction loss AND both
#                 critic inputs only see baseline ∩ (SIFT-warped) follow-up retina. Round 5's square outputs came
#                 from training on warped targets whose circle was clipped, unmasked.
#   LR            1e-5 for G and D (was 1e-4): fine-tuning wrecked round 5's pretrained Generator (FID 64 -> 149).
#   D_INIT_MODE   'full' = start from notebook 09's pretrained critic instead of a random one. If it fails to load,
#                 use 'reset-cls-head' (loads it, re-initialises the stage head).
#   SELECTION_MODE 'kid' = best epoch by KID vs the held-out real follow-ups; the pretrained Generator is scored
#                 as epoch 0 first, so if no epoch beats it best-G.ckpt IS the pretrained Generator. The KID of
#                 copying the baseline is printed once as a reference.
FOV_MASK = True
G_LR = D_LR = 1e-5
D_INIT_MODE = 'full'
SELECTION_MODE = 'kid'
# Round 7 (train_module2_poc.py docstring point 16) -- give the Generator a reason to change the grade:
#   LAMBDA_GRADER  0.5: the frozen EyePACS grader (CLS_CKPT) back in G's loss. Round 4 used 1.0 with lr 1e-4 and a random
#                  critic: the loss grader agreed 100 % at FID 266 (artefacts it read as lesions). Round 7 keeps round 6's
#                  lr 1e-5, pretrained critic and retina masks, halves the weight, and gates the epoch pick on KID (below).
#                  If KID still jumps (training log: 'not eligible (KID > ceiling)' every epoch), try 0.3.
#   LAMBDA_REC     5 (was 10): most training pairs don't progress, so a strong "match the follow-up" loss teaches copying.
#   SELECTION_MODE 'control-kid': an epoch is eligible only with stage control >= MIN_CONTROL (asked for one stage above
#                  the baseline, the grader reads that stage) AND KID <= KID_SLACK x the pretrained Generator's (epoch 0);
#                  the lowest KID among eligible epochs wins; none eligible -> best-G.ckpt = the pretrained Generator.
#   Stage control is graded by the EyePACS grader (the one in the loss): FGADR's kappa on Tianjin is ~0 and round 4's FGADR
#   stage control sat at ~2 % all run. FGADR's stage control is logged every epoch as a check, and the comparison
#   below reports both graders.
LAMBDA_GRADER = 0.5
LAMBDA_REC = 5.0
SELECTION_MODE = 'control-kid'
MIN_CONTROL = 0.2
KID_SLACK = 2.0
# Round 7 cascade clean-up (evaluation below + notebooks 08/10; training is unchanged): every step's colours matched to the
# real baseline (COLOR_MATCH; round 6's grey drift). FOV_ERODE / FOV_FEATHER pull the retina mask in / fade its edge
# (pixels at MODULE2_IMAGE_SIZE): 0, 0 = the baseline's full circle, as in round 6 -- 4, 3 (round 7's first run) made the
# eye ~12 % narrower with a dimmed rim. Same values in notebooks 08 and 10.
FOV_ERODE, FOV_FEATHER, COLOR_MATCH = 0, 0, True
FOV_ARG = ' --fov-mask' if FOV_MASK else ''  # not inline: a Python expression inside !python may not expand
CASCADE_ARGS = ((f' --fov-erode {FOV_ERODE} --fov-feather {FOV_FEATHER}' if FOV_MASK else '')
                + (' --color-match' if COLOR_MATCH else ''))
# Selection grader: round 7 (control-kid) = the EyePACS grader, FGADR logged as a check. Earlier rounds: with no grader in
# the loss, the EyePACS grader (CLS_CKPT, kappa 0.44 on Tianjin's clinical grades); with one, the other (INDEP_CKPT).
if SELECTION_MODE == 'control-kid':
    SEL_CKPT, SEL_NORM = CLS_CKPT, CLS_NORM
    CHECK_ARGS = (f' --selection-check-classifier-checkpoint "{INDEP_CKPT}" --selection-check-classifier-norm {INDEP_NORM}'
                  if INDEP_CKPT != CLS_CKPT else '')
else:
    SEL_CKPT, SEL_NORM = (CLS_CKPT, CLS_NORM) if LAMBDA_GRADER == 0 else (INDEP_CKPT, INDEP_NORM)
    CHECK_ARGS = ''
print('best-epoch grader:', SEL_CKPT, f'({SEL_NORM})', '| check grader:', INDEP_CKPT if CHECK_ARGS else None)
MODULE2_BATCH_SIZE = 8   # round 3's value; lower it if CUDA runs out of memory
MODULE2_RESUME_STATE = os.path.join(DRIVE_DATA_DIR, 'module2_runs', MODULE2_RUN_NAME, 'resume_state.pt')
%cd {REPO_CODE_DIR}
!python train_module2_poc.py --image-size {MODULE2_IMAGE_SIZE} \
    --fire-dir /content/data/FIRE_dataset \
    --longdr-dir /content/data/LongDRScreening_20150209 \
    --tianjin-dir "{TIANJIN_DIR}" \
    --fire-module1-cache "{MODULE1_CACHE_DIR}/module1_outputs_fire.pt" \
    --longdr-module1-cache "{MODULE1_CACHE_DIR}/module1_outputs_longdr.pt" \
    --tianjin-module1-cache "{MODULE1_CACHE_DIR}/module1_outputs_tianjin.pt" \
    --registration-cache "{REGISTRATION_CACHE}"{QUALITY_ARGS} \
    --num-epochs {MODULE2_EPOCHS} \
    --resume-state "{MODULE2_RESUME_STATE}" \
    --save-dir ./DRForestGAN-v2/stargan/{MODULE2_RUN_NAME}/{INIT_ARGS} \
    --sample-dir ./training_samples_poc/{MODULE2_RUN_NAME}/ \
    --selection-classifier-checkpoint "{SEL_CKPT}" --selection-classifier-norm {SEL_NORM}{CLS_NORM_ARGS} \
    --selection-alpha {SELECTION_ALPHA} --selection-mode {SELECTION_MODE} \
    --selection-min-control {MIN_CONTROL} --selection-kid-slack {KID_SLACK}{CHECK_ARGS} --lambda-rec {LAMBDA_REC} \
    --grader-loss-checkpoint "{CLS_CKPT}" --grader-loss-norm {CLS_NORM} --lambda-grader {LAMBDA_GRADER} \
    --diffaugment "{DIFFAUGMENT}" --lambda-drift {DRIFT} --batch-size {MODULE2_BATCH_SIZE} \
    --early-stop-patience {EARLY_STOP_PATIENCE} --tianjin-min-baseline-grade {MIN_BASELINE_GRADE} \
    --g-lr {G_LR} --d-lr {D_LR} --init-discriminator-mode {D_INIT_MODE}{FOV_ARG} \
    --num-workers 2 \
    --val-fraction 0.1 \
    --eval-batches 15 \
    --fid-n-bootstrap 10

best-epoch grader: /content/drive/MyDrive/Thesis_Datasets/module1_runs/drgnet_replication_prefinals/classification/eyepacs/saves/best_validation_weights.pt (eyepacs) | check grader: /content/drive/MyDrive/Thesis_Datasets/module1_runs/drgnet_replication_prefinals/classification/fgadr/saves/final_weights.pt
/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc
Classifier normalisation: eyepacs mean=[0.425753653049469, 0.29737451672554016, 0.21293757855892181] std=[0.27670302987098694, 0.20240527391433716, 0.1686241775751114]
Module 2 POC training (real Module 1 conditioning)
Device: cuda
fire_module1_cache_path: /content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5/module1_outputs_fire.pt
longdr_module1_cache_path: /content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5/module1_outputs_longdr.pt
tianjin_dir: /content/data/retinal-dr-longitudinal
tianjin_module1_cache_path: /content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_c

In [9]:
from IPython.display import Image as IPImage, display
import glob, json, os

# Sorted by epoch number (plain sorted() puts epoch_9.jpg after epoch_10.jpg ... and shows it as the last).
samples = sorted(glob.glob(f'{REPO_CODE_DIR}/training_samples_poc/{MODULE2_RUN_NAME}/epoch_*.jpg'),
                 key=lambda p: int(os.path.basename(p)[len('epoch_'):-len('.jpg')]))
if samples:
    display(IPImage(samples[-1]))
else:
    print(f'No training_samples_poc/{MODULE2_RUN_NAME}/epoch_*.jpg found -- the training cell above did not finish. Read its output.')

rp = f'{REPO_CODE_DIR}/DRForestGAN-v2/stargan/{MODULE2_RUN_NAME}/poc_results.json'
print(json.dumps(json.load(open(rp)), indent=2) if os.path.isfile(rp) else f'{rp} not found')

No training_samples_poc/models_poc_tianjin_from_nonlong_drgnet_comb_prefinals_r7/epoch_*.jpg found -- the training cell above did not finish. Read its output.
{
  "num_epochs": 40,
  "image_size": 128,
  "num_pairs": 671,
  "effective_samples": 2396,
  "g_lr": 1e-05,
  "d_lr": 1e-05,
  "beta1": 0.0,
  "beta2": 0.9,
  "n_critic": 5,
  "model_arch": "prefinals",
  "batch_size": 8,
  "lambda_cls": 1.0,
  "fire_module1_cache_path": "/content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5/module1_outputs_fire.pt",
  "longdr_module1_cache_path": "/content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5/module1_outputs_longdr.pt",
  "tianjin_dir": "/content/data/retinal-dr-longitudinal",
  "tianjin_module1_cache_path": "/content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5/module1_outputs_tianjin.pt",
  "registration_cache_path": "/content/drive/MyDrive/Thesis_Datasets/module2_runs/registration_prefinals_r5_sift/registered_fol

In [10]:
# Persist this run's checkpoints + poc_results.json to Drive (the runtime disk is wiped when the
# session ends) -- notebook 10 loads the Generator from here.
import shutil
_src = f'{REPO_CODE_DIR}/DRForestGAN-v2/stargan/{MODULE2_RUN_NAME}'
_dst = os.path.join(DRIVE_DATA_DIR, 'module2_runs', MODULE2_RUN_NAME)
# Per-epoch {epoch}-G.ckpt snapshots stay on the runtime (~33 MB each, not used downstream):
# Drive gets best-G.ckpt, final-G.ckpt, resume state and results only.
shutil.copytree(_src, _dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns('[0-9]*-G.ckpt'))
print('✓ Copied', _src, '->', _dst)
for _f in ('best-G.ckpt', 'final-G.ckpt'):
    _a, _b = os.path.join(_src, _f), os.path.join(_dst, _f)
    if os.path.isfile(_a):
        assert os.path.isfile(_b) and os.path.getsize(_b) == os.path.getsize(_a), (
            f'{_f} not fully saved to Drive ({_dst}) -- is your Google Drive full? Check drive.google.com/settings/storage and empty the Trash, then re-run this cell.')

✓ Copied /content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc/DRForestGAN-v2/stargan/models_poc_tianjin_from_nonlong_drgnet_comb_prefinals_r7 -> /content/drive/MyDrive/Thesis_Datasets/module2_runs/models_poc_tianjin_from_nonlong_drgnet_comb_prefinals_r7


In [11]:
# Per-cascade-step FID/PSNR/SSIM against Tianjin's real follow-up images (the only source with
# a real grade on both baseline and follow-up -- FIRE/LongDR can't participate, see
# evaluate_trajectory.py's module docstring), plus Module 1's own re-grading consistency at
# every cascade step (generate, re-segment, re-grade, feed forward). The single-step metrics
# from the training cell above don't report this.
%cd {REPO_CODE_DIR}
_extra_seg_args = ''
if os.path.isfile(HE_CKPT):
    _extra_seg_args += f' --seg-checkpoint HE={HE_CKPT}'
if os.path.isfile(SE_CKPT):
    _extra_seg_args += f' --seg-checkpoint SE={SE_CKPT}'

# Evaluates the best-epoch Generator (best-G.ckpt, picked on held-out pairs). Falls back to
# final-G.ckpt (last epoch; also the only one with --val-fraction 0), then -- if the training
# cell got cut short before writing either -- to the latest numbered {epoch}-G.ckpt, and says so.
GEN_SAVE_DIR = f'{REPO_CODE_DIR}/DRForestGAN-v2/stargan/{MODULE2_RUN_NAME}'
_best = os.path.join(GEN_SAVE_DIR, 'best-G.ckpt')
_final = os.path.join(GEN_SAVE_DIR, 'final-G.ckpt')
if os.path.isfile(_final) and os.path.isfile(_best):
    GENERATOR_CKPT = _best
elif os.path.isfile(_final):
    GENERATOR_CKPT = _final
else:
    _epoch_ckpts = glob.glob(os.path.join(GEN_SAVE_DIR, '*-G.ckpt'))
    assert _epoch_ckpts, f"No checkpoint at all in {GEN_SAVE_DIR} -- the training cell above did not finish a single epoch."
    GENERATOR_CKPT = max(_epoch_ckpts, key=lambda p: int(os.path.basename(p).split('-')[0]))
    print(f"⚠ final-G.ckpt not found -- training was likely interrupted. Using {GENERATOR_CKPT} instead "
          "(an earlier epoch, not the full run).")

# Round 4 (item R4-2.5): only Tianjin patients the Generator never trained on (--exclude-train-from: every
# patient with a training pair, from poc_results.json), re-graded by the INDEPENDENT grader (not the one in
# G's loss). Three runs, saved on Drive next to the run:
#   trajectory_eval_results.json             primary: unseen patients, independent grader, every baseline grade
#   trajectory_eval_results_in_scope.json    the same, baseline >= mild NPDR (the manuscript's scope)
#   trajectory_eval_results_loss_grader.json the grader G was optimised on -- optimistic, report only next to the primary
POC_RESULTS = os.path.join(GEN_SAVE_DIR, 'poc_results.json')
_norm = lambda n: ' --classifier-norm eyepacs' if n == 'eyepacs' else ''  # noqa: E731
TRAJ_EVAL_JSON = os.path.join(DRIVE_DATA_DIR, 'module2_runs', MODULE2_RUN_NAME, 'trajectory_eval_results.json')
TRAJ_EVAL_SCOPE_JSON = TRAJ_EVAL_JSON.replace('.json', '_in_scope.json')
TRAJ_EVAL_LOSSGRADER_JSON = TRAJ_EVAL_JSON.replace('.json', '_loss_grader.json')
# Round 7: the EyePACS grader (CLS_CKPT, kappa 0.44 on Tianjin) is the primary grader even though it is in G's loss --
# FGADR's kappa on Tianjin is ~0, so it can't be the main yardstick. Every JSON also carries FGADR's numbers
# (--second-classifier-checkpoint -> the "final_step" ... "second" keys), so a gain only counts if FGADR sees it too
# (or is reported as unconfirmed where FGADR can't read real progression either -- see the comparison below).
# Two runs (the earlier third run with the other grader is now inside every JSON):
#   trajectory_eval_results.json            unseen patients, every baseline grade (+ FIRE/LongDR self-consistency)
#   trajectory_eval_results_in_scope.json   the same, baseline >= mild NPDR (the manuscript's scope)
EVAL_CKPT, EVAL_NORM, OTHER_CKPT, OTHER_NORM = CLS_CKPT, CLS_NORM, INDEP_CKPT, INDEP_NORM
SECOND_ARGS = (f' --second-classifier-checkpoint "{OTHER_CKPT}" --second-classifier-norm {OTHER_NORM}'
               if OTHER_CKPT != EVAL_CKPT else '')
_cls_arg = _norm(EVAL_NORM)  # not inline: a Python call inside !python is not expanded
for _out, _extra in ((TRAJ_EVAL_JSON, ' --self-consistency-dirs fire=/content/data/FIRE_dataset'
                                      ' --self-consistency-dirs longdr=/content/data/LongDRScreening_20150209'),
                     (TRAJ_EVAL_SCOPE_JSON, ' --min-baseline-grade 1')):
    !python evaluate_trajectory.py --image-size {MODULE2_IMAGE_SIZE} \
        --tianjin-dir "{TIANJIN_DIR}" \
        --tianjin-module1-cache "{MODULE1_CACHE_DIR}/module1_outputs_tianjin.pt" \
        --generator-checkpoint "{GENERATOR_CKPT}" \
        --classifier-checkpoint "{EVAL_CKPT}"{_cls_arg}{SECOND_ARGS} \
        --seg-checkpoint EX="{EX_CKPT}" --seg-checkpoint MA="{MA_CKPT}"{_extra_seg_args} \
        --exclude-train-from "{POC_RESULTS}"{_extra}{FOV_ARG}{CASCADE_ARGS} \
        --out "{_out}"

import json as _json
print(_json.dumps(_json.load(open(TRAJ_EVAL_JSON)), indent=2))
_r = _json.load(open(TRAJ_EVAL_SCOPE_JSON))
print('in scope (baseline >= 1)', {k: (v['n'], round(v['module1_agrees_with_target_stage_rate'], 3))
                                   for k, v in _r['module1_consistency'].items()}, '(n, agreement)')


/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc
Classifier normalisation: eyepacs mean=[0.425753653049469, 0.29737451672554016, 0.21293757855892181] std=[0.27670302987098694, 0.20240527391433716, 0.1686241775751114]
Lesion segmentation: tiled (retina 1280 px, 512 px tiles, stride 384) | retina crop: on
Device: cuda

[1/4] Loading Tianjin survival dataset (source of real baseline+follow-up grade pairs)...
✓ Loaded Module 1 cache: 1234 images (/content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5/module1_outputs_tianjin.pt)
  Total eye-pairs in corrected_manifest.csv: 1115
  Baseline sheet 'Baseline': patient_id <- 'ID', OS <- 'OS Grade（1=No Apparent Retinopathy；2=Mild Non-Proliferative Diabetic Retinopathy；3=Moderate Non-Proliferative Diabetic Retinopathy；4=Severe Non-Proliferative Diabetic Retinopathy；5=Proliferative Diabetic Retinopathy；6=After laser；7=Missing）', OD <- 'OD Grade（1=No Apparent Retinopathy；2=Mild Non-Proliferative Diabetic Retinopa

In [12]:
# Item 5.4: the EyePACS-pretrained Generator BEFORE fine-tuning ("epoch 0" -- notebook 09's
# final-G.ckpt), through the same evaluation as above, so the effect of the longitudinal fine-tuning
# on per-step quality and Module 1 agreement can be read directly. Saved next to the run.
# Round 6: off -- the five-row comparison below scores the pretrained Generator on the same patients as every
# other row (and with the same retina masking). Set True for the earlier per-run 'epoch 0' JSON as well.
EVAL_EPOCH0 = False
if EVAL_EPOCH0 and USE_NONLONG_PRETRAIN:
    TRAJ_EVAL_EPOCH0_JSON = os.path.join(DRIVE_DATA_DIR, 'module2_runs', MODULE2_RUN_NAME,
                                         'trajectory_eval_results_epoch0_pretrained.json')
    %cd {REPO_CODE_DIR}
    _cls_arg0 = _norm(EVAL_NORM)
    !python evaluate_trajectory.py --image-size {MODULE2_IMAGE_SIZE} \
        --tianjin-dir "{TIANJIN_DIR}" \
        --tianjin-module1-cache "{MODULE1_CACHE_DIR}/module1_outputs_tianjin.pt" \
        --generator-checkpoint "{NONLONG_DIR}/final-G.ckpt" \
        --classifier-checkpoint "{EVAL_CKPT}"{_cls_arg0} \
        --seg-checkpoint EX="{EX_CKPT}" --seg-checkpoint MA="{MA_CKPT}"{_extra_seg_args} \
        --exclude-train-from "{POC_RESULTS}"{FOV_ARG}{CASCADE_ARGS}{SECOND_ARGS} \
        --out "{TRAJ_EVAL_EPOCH0_JSON}"
    _e0, _ft = _json.load(open(TRAJ_EVAL_EPOCH0_JSON)), _json.load(open(TRAJ_EVAL_JSON))
    for _k in ('quality', 'module1_consistency'):
        print(_k, '\n  epoch 0   :', _e0.get(_k), '\n  fine-tuned:', _ft.get(_k))
else:
    print('Skipped (EVAL_EPOCH0 = False or USE_NONLONG_PRETRAIN = False -- no pretrained Generator).')

Skipped (EVAL_EPOCH0 = False or USE_NONLONG_PRETRAIN = False -- no pretrained Generator).


### Round 7: the same unseen patients, two graders, a ceiling and a floor

Every row runs through the same evaluation on Tianjin patients that **no** round could have trained on (`--exclude-scope module2`), with the same retina mask, edge fade and colour match (`FOV_ARG` + `CASCADE_ARGS`). The numbers that decide the round are graded on each pair's **last** image (the one compared with the real follow-up), by **both** graders:

1. **copy the baseline** -- no Generator; every stage is the baseline photo (the floor)
2. **real follow-up photo** -- no Generator; the real 2-year photo (the ceiling: how well each grader reads real progression)
3. **round 5 pretrained** Generator (notebook 09, before any fine-tuning)
4. **round 6** fine-tuned Generator
5. **round 7** fine-tuned Generator (this run)
6. **round 7, every stage from the real baseline** instead of from the previous synthesized step (does feeding G's own output back in add the blur?)

Columns: **agrees** = the grader reads the last image as the real follow-up grade; **grade up** = the grader rates the last image higher than it rates the real baseline (relative to the grader's own reading, so a grader that over- or under-grades Tianjin doesn't count). Brackets are 95 % intervals -- n is small, so overlapping intervals mean "no clear difference". `step` = how many stages the pair progressed; `all` pools them. KID/SSIM per step as before (none for the real follow-up row).

Round 7 trains G against the EyePACS grader, so it can raise the EyePACS numbers with artefacts; a gain counts only if FGADR sees it too. If the ceiling row shows FGADR can't read real progression either, FGADR can't confirm anything and the verdict says so -- then check the notebook 08/10 figures by eye.


In [13]:
# Round 7: six rows on the SAME unseen patients (--exclude-scope module2), graded by both graders. Results (JSON + CSV)
# are saved on Drive in <this run>/round7_comparison/ and reused on a re-run (delete a JSON to recompute that row).
# A JSON from an older evaluate_trajectory.py (eval_version below EVAL_VERSION) or with different FOV_ERODE / FOV_FEATHER /
# COLOR_MATCH settings is recomputed automatically -- the first round-7 run's rows (mask-edge bug, raw-baseline grading,
# 4/3 mask) are redone without deleting anything by hand.
EVAL_VERSION = 2
import json as _json
import pandas as pd
R5_PRETRAINED_CKPT = os.path.join(NONLONG_DIR, 'final-G.ckpt')
R6_CKPT = os.path.join(DRIVE_DATA_DIR, 'module2_runs', 'models_poc_tianjin_from_nonlong' + BASE_RUN_TAG + '_r6', 'best-G.ckpt')
CMP_DIR = os.path.join(DRIVE_DATA_DIR, 'module2_runs', MODULE2_RUN_NAME, 'round7_comparison')
os.makedirs(CMP_DIR, exist_ok=True)
# (tag, how, checkpoint, label) -- how: 'copy' | 'real' | 'cascade' | 'direct'
CMP_ROWS = [('copy_baseline', 'copy', None, 'copy the baseline (control)'),
            ('real_followup', 'real', None, 'real follow-up photo (ceiling)'),
            ('round5_pretrained', 'cascade', R5_PRETRAINED_CKPT, 'round 5 pretrained (no fine-tuning)'),
            ('round6', 'cascade', R6_CKPT, 'round 6 fine-tuned'),
            ('round7', 'cascade', GENERATOR_CKPT, 'round 7 fine-tuned (this run)'),
            ('round7_direct', 'direct', GENERATOR_CKPT, 'round 7, every stage from the real baseline')]
%cd {REPO_CODE_DIR}
_cls_cmp = _norm(EVAL_NORM)  # not inline: a Python call inside !python is not expanded
for _tag, _how, _ckpt, _label in CMP_ROWS:
    _out = os.path.join(CMP_DIR, f'trajectory_eval_{_tag}.json')
    if os.path.isfile(_out):
        _old = _json.load(open(_out))
        _same = (_old.get('eval_version', 1) >= EVAL_VERSION and _old.get('fov_erode', 0) == FOV_ERODE
                 and _old.get('fov_feather', 0) == FOV_FEATHER and bool(_old.get('color_match')) == COLOR_MATCH)
        if _same:
            print('✓ Reusing', _out)
            continue
        print('↻ Recomputing (older evaluation or different mask/colour settings):', _out)
    if _ckpt is not None and not os.path.isfile(_ckpt):
        print(f'⚠ {_label}: checkpoint not found ({_ckpt}) -- row skipped')
        continue
    _gen_arg = {'copy': ' --copy-baseline', 'real': ' --real-followup'}.get(_how, f' --generator-checkpoint "{_ckpt}"')
    if _how == 'direct':
        _gen_arg += ' --cascade-mode direct'
    print(f'\n=== {_label} ===')
    !DRP_RETINA_CROP=1 python evaluate_trajectory.py --image-size {MODULE2_IMAGE_SIZE} \
        --tianjin-dir "{TIANJIN_DIR}" \
        --tianjin-module1-cache "{MODULE1_CACHE_DIR}/module1_outputs_tianjin.pt"{_gen_arg} \
        --classifier-checkpoint "{EVAL_CKPT}"{_cls_cmp}{SECOND_ARGS} \
        --seg-checkpoint EX="{EX_CKPT}" --seg-checkpoint MA="{MA_CKPT}"{_extra_seg_args} \
        --exclude-train-from "{POC_RESULTS}" --exclude-scope module2{FOV_ARG}{CASCADE_ARGS} \
        --out "{_out}"

_res, _held = {}, None
for _tag, _how, _ckpt, _label in CMP_ROWS:
    _out = os.path.join(CMP_DIR, f'trajectory_eval_{_tag}.json')
    if os.path.isfile(_out):
        _res[_tag] = _json.load(open(_out))
        _held = _held or _res[_tag].get('held_out')
G1, G2 = CLASSIFIER_RUN, INDEP_RUN  # grader names for the column headers
GRADERS = ['primary'] + (['second'] if SECOND_ARGS else [])
NAMES = {'primary': G1, 'second': G2}

def _rate(e, key):
    if not e or not e.get('n'):
        return None
    lo, hi = e[f'{key}_ci95']
    return f"{e[f'{key}_rate']:.0%} [{lo:.0%}-{hi:.0%}]"

_rows = []
for _tag, _how, _ckpt, _label in CMP_ROWS:
    _r = _res.get(_tag)
    if not _r:
        continue
    for _k in sorted(_r.get('final_step', {}), key=lambda k: (k == 'all', k)):
        _fs, _q = _r['final_step'][_k], _r['quality'].get(_k) or {}
        _row = {'Generator': _label, 'step': _k.replace('step_', ''), 'n': _fs['n'],
                'KID': None if _q.get('kid_mean') is None else round(_q['kid_mean'], 4),
                'KID std': None if _q.get('kid_std') is None else round(_q['kid_std'], 4),
                'SSIM': None if _q.get('mean_ssim') is None else round(_q['mean_ssim'], 3)}
        for _g in GRADERS:
            _row[f'agrees ({NAMES[_g]})'] = _rate(_fs.get(_g), 'agree')
            _row[f'grade up ({NAMES[_g]})'] = _rate(_fs.get(_g), 'went_up')
        _rows.append(_row)
print('Patients:', _held)
print(f'Graders: {EVAL_CKPT} ({EVAL_NORM})' + (f' + {OTHER_CKPT} ({OTHER_NORM})' if SECOND_ARGS else '')
      + f' | retina mask: {FOV_MASK} (erode {FOV_ERODE}, feather {FOV_FEATHER}) | colour match: {COLOR_MATCH}')
if _rows:
    _df = pd.DataFrame(_rows)
    _df.to_csv(os.path.join(CMP_DIR, 'round7_comparison.csv'), index=False)
    display(_df)

# ---- Verdict (pooled 'all' bucket for the grade numbers; step 1 for KID, the only step with enough pairs) ----
def _fs(tag, g, key='all'):
    return ((_res.get(tag) or {}).get('final_step', {}).get(key) or {}).get(g)
def _q(tag, key='step_1'):
    return (_res.get(tag) or {}).get('quality', {}).get(key)
def _informative(g):
    """Does this grader see real progression at all? The real follow-ups' grade-up interval sits entirely above the
    copy row's (non-overlapping 95 % intervals)."""
    real, copy = _fs('real_followup', g), _fs('copy_baseline', g)
    if not (real and copy and real['n'] and copy['n']):
        return None
    return real['went_up_ci95'][0] > copy['went_up_ci95'][1]
def _progress(g):
    """Round 7 vs the pretrained Generator: grade goes up clearly more often, and exact agreement beats copying."""
    r7, pre, copy = _fs('round7', g), _fs('round5_pretrained', g), _fs('copy_baseline', g)
    if not (r7 and pre and copy and r7['n']):
        return None
    return (r7['went_up_rate'] - pre['went_up_rate'] >= 0.10) and (r7['agree_rate'] >= copy['agree_rate'] + 0.05)
def _kid_ok(a, b):
    return bool(a and b and a['kid_mean'] is not None and b['kid_mean'] is not None
                and a['kid_mean'] <= b['kid_mean'] + 2 * max(a['kid_std'] or 0.0, b['kid_std'] or 0.0))

print('\nCan each grader see real progression? (ceiling row vs copy row, grade-up rate)')
for _g in GRADERS:
    real, copy = _fs('real_followup', _g), _fs('copy_baseline', _g)
    if real and copy:
        print(f"  {NAMES[_g]}: real follow-ups {real['went_up_rate']:.0%} vs copy {copy['went_up_rate']:.0%} -> "
              + ('yes' if _informative(_g) else 'NO -- this grader cannot confirm or rule out progression'))
try:
    _poc = _json.load(open(POC_RESULTS))
    print(f"\nTraining: best epoch {_poc.get('best_epoch')} | passed the stage-control/KID gate: "
          f"{_poc.get('best_passed_selection_gate')} | stage control {_poc.get('best_stage_control')} | "
          f"KID ceiling {_poc.get('selection_kid_ceiling')}")
except (OSError, ValueError):
    pass

p1 = _progress('primary')
p2 = _progress('second') if 'second' in GRADERS else None
inf2 = _informative('second') if 'second' in GRADERS else None
kid_ok = _kid_ok(_q('round7'), _q('round5_pretrained'))
print('\nVerdict:')
if p1 is None:
    print('  incomplete -- a row is missing (read the outputs above)')
elif p1 and not kid_ok:
    print(f"  GRADE MOVES, IMAGES DEGRADE: {G1} reads more progression, but round 7's step-1 KID "
          f"({_q('round7')['kid_mean']:.4f}) is well above the pretrained Generator's ({_q('round5_pretrained')['kid_mean']:.4f}). "
          'Likely artefacts -- lower LAMBDA_GRADER (e.g. 0.3) and re-run.')
elif p1 and p2:
    print(f'  PROGRESSING: round 7 raises the grade on BOTH graders ({G1} and {G2}) with KID close to the pretrained '
          "Generator's. Use GENERATOR = 'finetuned' in notebooks 08 and 10.")
elif p1 and inf2:
    print(f'  {G1.upper()} ONLY -- probably artefacts: {G2} can see real progression (ceiling row) but not round 7\'s. '
          'Look at the notebook 08/10 figures for coloured marks instead of lesions; try a lower LAMBDA_GRADER.')
elif p1:
    print(f'  PROGRESSING ON {G1.upper()}, UNCONFIRMED: {G2} cannot read real progression either (ceiling row), so it '
          'cannot confirm. Check the notebook 08/10 figures by eye for plausible lesions before reporting it.')
else:
    _r7, _pre = _fs('round7', 'primary'), _fs('round5_pretrained', 'primary')
    print(f"  NOT PROGRESSING YET: on {G1}, grade went up {_r7['went_up_rate']:.0%} (round 7) vs {_pre['went_up_rate']:.0%} "
          f"(pretrained); exact agreement {_r7['agree_rate']:.0%} vs copy {_fs('copy_baseline', 'primary')['agree_rate']:.0%}. "
          'Report that Module 2 does not yet produce grade progression.')
# Cascade vs direct (step 2: the first step where the cascade feeds G's own output back in)
_c2, _d2 = _q('round7', 'step_2'), _q('round7_direct', 'step_2')
if _c2 and _d2 and _c2['kid_mean'] is not None and _d2['kid_mean'] is not None:
    _gap = _c2['kid_mean'] - _d2['kid_mean']
    _tol = max(_c2['kid_std'] or 0.0, _d2['kid_std'] or 0.0)
    print(f"Cascade vs direct at step 2: KID {_c2['kid_mean']:.4f} vs {_d2['kid_mean']:.4f} -> "
          + ('feeding G\'s output back in adds damage' if _gap > _tol else
             'no clear difference -- the blur is not from the cascade feedback'))


/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc
↻ Recomputing (older evaluation or different mask/colour settings): /content/drive/MyDrive/Thesis_Datasets/module2_runs/models_poc_tianjin_from_nonlong_drgnet_comb_prefinals_r7/round7_comparison/trajectory_eval_copy_baseline.json

=== copy the baseline (control) ===
Classifier normalisation: eyepacs mean=[0.425753653049469, 0.29737451672554016, 0.21293757855892181] std=[0.27670302987098694, 0.20240527391433716, 0.1686241775751114]
Lesion segmentation: tiled (retina 1280 px, 512 px tiles, stride 384) | retina crop: on
Device: cuda

[1/4] Loading Tianjin survival dataset (source of real baseline+follow-up grade pairs)...
✓ Loaded Module 1 cache: 1234 images (/content/drive/MyDrive/Thesis_Datasets/module1_cache_drgnet_comb_prefinals_r5/module1_outputs_tianjin.pt)
  Total eye-pairs in corrected_manifest.csv: 1115
  Baseline sheet 'Baseline': patient_id <- 'ID', OS <- 'OS Grade（1=No Apparent Retinopathy；2=Mild Non-Proliferative 

,Generator,step,n,KID,KID std,SSIM,agrees (eyepacs),grade up (eyepacs),agrees (fgadr),grade up (fgadr)
0,copy the baseline (control),1,54,0.0217,0.0042,0.658,7% [3%-18%],0% [0%-7%],9% [4%-20%],0% [0%-7%]
1,copy the baseline (control),2,29,0.0183,0.0053,0.643,28% [15%-46%],0% [0%-12%],14% [5%-31%],0% [0%-12%]
2,copy the baseline (control),all,83,NaN,NaN,NaN,14% [8%-24%],0% [0%-4%],11% [6%-19%],0% [0%-4%]
3,real follow-up photo (ceiling),1,54,NaN,NaN,NaN,13% [6%-24%],15% [8%-27%],7% [3%-18%],9% [4%-20%]
4,real follow-up photo (ceiling),2,29,NaN,NaN,NaN,34% [20%-53%],24% [12%-42%],14% [5%-31%],3% [1%-17%]
5,real follow-up photo (ceiling),all,83,NaN,NaN,NaN,20% [13%-30%],18% [11%-28%],10% [5%-18%],7% [3%-15%]
6,round 5 pretrained (no fine-tuning),1,54,0.0309,0.0046,0.655,9% [4%-20%],17% [9%-29%],7% [3%-18%],11% [5%-22%]
7,round 5 pretrained (no fine-tuning),2,29,0.0360,0.0067,0.633,21% [10%-38%],17% [8%-35%],14% [5%-31%],7% [2%-22%]
8,round 5 pretrained (no fine-tuning),all,83,NaN,NaN,NaN,13% [8%-22%],17% [10%-26%],10% [5%-18%],10% [5%-18%]
9,round 6 fine-tuned,1,54,0.0252,0.0037,0.654,4% [1%-13%],13% [6%-24%],7% [3%-18%],9% [4%-20%]



Can each grader see real progression? (ceiling row vs copy row, grade-up rate)
  eyepacs: real follow-ups 18% vs copy 0% -> yes
  fgadr: real follow-ups 7% vs copy 0% -> NO -- this grader cannot confirm or rule out progression

Training: best epoch 8 | passed the stage-control/KID gate: True | stage control 0.2777777777777778 | KID ceiling 0.02728899009525776

Verdict:
  NOT PROGRESSING YET: on eyepacs, grade went up 45% (round 7) vs 17% (pretrained); exact agreement 12% vs copy 14%. Report that Module 2 does not yet produce grade progression.
Cascade vs direct at step 2: KID 0.0583 vs 0.0337 -> feeding G's output back in adds damage


In [14]:
# Task L/N/O figures: per-cascade-step FID/PSNR/SSIM, G/D/reconstruction loss curves,
# and Module 1 re-grading consistency AUC (with DRForecastGAN's 0.87/0.85 reference line).
# FIRE/LongDR self-consistency numbers above are tagged 'self-consistency -- not validated
# against ground truth' in trajectory_eval_results.json -- never read them next to the
# Tianjin numbers as the same kind of claim.
import os
FIG_DIR = f'/content/drive/MyDrive/Thesis_Datasets/module2_runs/figures{RUN_TAG}/{MODULE2_RUN_NAME}'  # per run (ablation)
os.makedirs(FIG_DIR, exist_ok=True)

%cd {REPO_CODE_DIR}
!python plot_module2_figures.py per-step-quality \
    --trajectory-eval-json "{TRAJ_EVAL_JSON}" \
    --out {FIG_DIR}/module2_per_step_quality.png
!python plot_module2_figures.py loss-curves \
    --poc-results-json {GEN_SAVE_DIR}/poc_results.json \
    --out {FIG_DIR}/module2_loss_curves.png
!python plot_module2_figures.py consistency-auc \
    --trajectory-eval-json "{TRAJ_EVAL_JSON}" \
    --out {FIG_DIR}/module2_consistency_auc.png


/content/DRProgression/M2-DRProgression-VerM-module1-fgadr-poc
  ⚠ an FID range does not contain its point estimate -- error bar clipped at the point
✓ Wrote /content/drive/MyDrive/Thesis_Datasets/module2_runs/figures_drgnet_comb_prefinals_r7/models_poc_tianjin_from_nonlong_drgnet_comb_prefinals_r7/module2_per_step_quality.png
✓ Wrote /content/drive/MyDrive/Thesis_Datasets/module2_runs/figures_drgnet_comb_prefinals_r7/models_poc_tianjin_from_nonlong_drgnet_comb_prefinals_r7/module2_loss_curves.png
✓ Wrote /content/drive/MyDrive/Thesis_Datasets/module2_runs/figures_drgnet_comb_prefinals_r7/models_poc_tianjin_from_nonlong_drgnet_comb_prefinals_r7/module2_consistency_auc.png


## Caveats

- **Round 7:** the EyePACS grader is in G's loss AND is the primary grader (FGADR's kappa on Tianjin is ~0), so its
  agreement is optimistic by construction. Read it next to FGADR's columns and the real-follow-up ceiling row; the
  verdict says when FGADR can't confirm. The training-time stage control is graded by the same EyePACS grader.

- **Round 4:** every cascade number in `trajectory_eval_results*.json` is on Tianjin patients the Generator
  never trained on (`evaluated_on` / `held_out` in the JSON), graded by the grader that is NOT in the
  Generator's loss. Report that one; `_loss_grader.json` (the grader G optimised) is optimistic by
  construction. FID is only computed for steps with >= 50 pairs -- report KID (unbiased at any n) with n.
- Compare the cascade agreement with notebook 04_05's grader-resolution check: the grader's agreement on
  REAL images shrunk to `MODULE2_IMAGE_SIZE` is the ceiling for this metric.

- MODULE2_EPOCHS = 100 epochs (up from notebook 04's 5), Adam settings tuned to lr=2e-4, β1=0.0,
  β2=0.9, EX+MA-only segmentation conditioning by default -- HE/SE are picked up automatically
  once trained).
- Tianjin adds a real clinical grade conditioning source, resolved per-eye. The
  ETDRS(1-5)->ICDR(0-4) offset is a numeric relabeling, not a claim that ETDRS and ICDR are
  clinically equivalent staging criteria (see `tianjin_dataset.py`).
- FIRE and LongDR still fall back to placeholder Stage-2 conditioning wherever no Module 1
  cache entry exists for a given image -- this run does not change that.
- AdaIN stage conditioning, a training-only registration network, and autoregressive
  multi-stage cascade generation run on top of the DRForecastGAN base. The cascade follows a
  4-step loop per stage: generate, re-segment with Module 1, re-grade with Module 1 as a
  consistency check, feed forward.
- The per-cascade-step numbers above come from very few real Tianjin progression pairs at POC
  scale; read the n_pairs in each bucket before treating any step's FID/PSNR/SSIM as reliable.
  Buckets with too few pairs report FID as null rather than a misleading number.
- `module1_consistency` (does the synthesized image at step N read as stage N to Module 1's
  own grader?) is a separate signal from PSNR/SSIM/FID -- a low rate could mean the generator
  produced something implausible, or that Module 1's grader is unreliable on
  out-of-distribution synthesized images, or both.
- The registration network's warp quality has not been visually audited here -- inspect a few
  `registered_followups.pt` entries before trusting that pre-registration measurably helped.
- Module 3 (not trained in this notebook) also conditions on the baseline eye's own severity
  grade, not just the LBS stratum -- see notebook 07.

## Before closing the runtime

Files written to Drive upload in the background. Run this last cell and wait for the ✓ before
disconnecting or deleting the runtime -- otherwise large files written at the end (model weights,
caches) can be lost.

In [15]:
from google.colab import drive
drive.flush_and_unmount()
print('✓ Everything is uploaded to Google Drive -- safe to disconnect the runtime.')

✓ Everything is uploaded to Google Drive -- safe to disconnect the runtime.
